In [2]:
import os
import glob
import tiktoken
import numpy as np
import ollama  
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

/var/folders/z8/6vbwx_z93ss21cqr59dbflfh0000gn/T/ipykernel_26117/424697612.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [4]:
MODEL = "qwen3:8b"
db_name ="vector_db"
print (f"Using loacl model OLlama : {MODEL}")

Using loacl model OLlama : qwen3:8b


In [5]:
knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowlege base")

entire_knowledge_base = ""
for file_path in files:
    with open(file_path,"r", encoding="utf-8") as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"
        print (f"Total characters in knowledge base:{len(entire_knowledge_base):,}")

Found 76 files in the knowlege base
Total characters in knowledge base:3,709
Total characters in knowledge base:9,067
Total characters in knowledge base:14,040
Total characters in knowledge base:18,189
Total characters in knowledge base:22,621
Total characters in knowledge base:26,795
Total characters in knowledge base:30,787
Total characters in knowledge base:34,922
Total characters in knowledge base:37,808
Total characters in knowledge base:41,468
Total characters in knowledge base:54,405
Total characters in knowledge base:59,759
Total characters in knowledge base:63,169
Total characters in knowledge base:65,697
Total characters in knowledge base:70,813
Total characters in knowledge base:79,036
Total characters in knowledge base:85,340
Total characters in knowledge base:88,325
Total characters in knowledge base:95,223
Total characters in knowledge base:98,021
Total characters in knowledge base:101,427
Total characters in knowledge base:107,206
Total characters in knowledge base:110,3

In [42]:
encoding = tiktoken.get_encoding("cl100k_base")
tokens = encoding.encode(entire_knowledge_base)
token_counts = len(tokens)
print(f"Approximate token count:{token_counts}")

Approximate token count:63721


In [51]:
folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
     doc_type = os.path.basename(folder)
     loader = DirectoryLoader(
          folder,
          glob="**/*.md",
          loader_cls=TextLoader,
          loader_kwargs={"encoding":"utf-8"},

     )
     folders_docs = loader.load()
     for doc in folders_docs:
          doc.metadata["doc_type"]=doc_type
          documents.append(doc)
          print(f"Loaded {len(documents)} documents")

Loaded 1 documents
Loaded 2 documents
Loaded 3 documents
Loaded 4 documents
Loaded 5 documents
Loaded 6 documents
Loaded 7 documents
Loaded 8 documents
Loaded 9 documents
Loaded 10 documents
Loaded 11 documents
Loaded 12 documents
Loaded 13 documents
Loaded 14 documents
Loaded 15 documents
Loaded 16 documents
Loaded 17 documents
Loaded 18 documents
Loaded 19 documents
Loaded 20 documents
Loaded 21 documents
Loaded 22 documents
Loaded 23 documents
Loaded 24 documents
Loaded 25 documents
Loaded 26 documents
Loaded 27 documents
Loaded 28 documents
Loaded 29 documents
Loaded 30 documents
Loaded 31 documents
Loaded 32 documents
Loaded 33 documents
Loaded 34 documents
Loaded 35 documents
Loaded 36 documents
Loaded 37 documents
Loaded 38 documents
Loaded 39 documents
Loaded 40 documents
Loaded 41 documents
Loaded 42 documents
Loaded 43 documents
Loaded 44 documents
Loaded 45 documents
Loaded 46 documents
Loaded 47 documents
Loaded 48 documents
Loaded 49 documents
Loaded 50 documents
Loaded 51

In [52]:
documents[1]

Document(metadata={'source': 'knowledge-base/products/Claimllm.md', 'doc_type': 'products'}, page_content="# Product Summary\n\n# Claimllm\n\n## Summary\n\nClaimllm is Insurellm's revolutionary claims processing platform that transforms the claims experience for insurers, adjusters, and policyholders. Powered by advanced AI, machine learning, and computer vision, Claimllm automates claims handling across all insurance lines—from first notice of loss through final settlement. By dramatically reducing processing time, improving accuracy, and enhancing fraud detection, Claimllm enables insurers to deliver exceptional claims service while significantly reducing operational costs. The platform seamlessly integrates with existing policy administration and core systems to create a unified insurance ecosystem.\n\n## Features\n\n### 1. Intelligent FNOL Processing\nClaimllm's AI-powered first notice of loss intake captures claim details through multiple channels including mobile apps, web portal

In [53]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)
print(f"Divided into {len(chunks)} chunks")
print (f"First chunk\n\n{chunks[0]}")

Divided into 413 chunks
First chunk

page_content='# Product Summary

# Rellm: AI-Powered Enterprise Reinsurance Solution

## Summary

Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.

## Features

### AI-Driven Analytics
Rellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.' metadata={'source': 'knowledge-base/products/Rellm.md', 'd

In [54]:
chunks[100]

Document(metadata={'source': 'knowledge-base/contracts/Contract with National Claims Network for Claimllm.md', 'doc_type': 'contracts'}, page_content="7. **Business Continuity:** Insurellm provides disaster recovery with 4-hour RTO (Recovery Time Objective) and 1-hour RPO (Recovery Point Objective).\n\n---\n\n## Renewal\n\nThis agreement includes a mutual 120-day renewal notice period. National Claims Network receives guaranteed enterprise pricing for renewal equal to or better than new enterprise customers at renewal time. Contract may be extended in 12-month increments with mutual written agreement.\n\n---\n\n## Features\n\nNational Claims Network will receive the complete Claimllm Enterprise suite:\n\n1. **Unlimited Claims Processing:** No volume restrictions, supporting National's processing of 100,000+ claims annually with scalability to 500,000+ claims as business grows.\n\n2. **White-Label Platform:** Complete branding customization including:\n   - Custom domain names (claims.n

In [55]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectorstore created with 413 documents


In [56]:
collection = vectorstore._collection
count = collection.count()
 
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")
 


There are 413 vectors with 384 dimensions in the vector store


In [58]:
result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
chroma_documents = result["documents"]
metadatas = result["metadatas"]
doc_types = [metadata["doc_type"] for metadata in metadatas]

In [59]:
category_list = ["products", "employees", "contracts", "company"]
color_list = ["blue", "green", "red", "orange"]
colors = [color_list[category_list.index(t)] for t in doc_types]

In [61]:
tsne = TSNE(n_components=2, random_state=42)
reduce_vectors = tsne.fit_transform(vectors)
fig = go.Figure(
    data=[
        go.Scatter(
            x=reduce_vectors[:, 0],
            y=reduce_vectors[:, 1],
            mode="markers",
            marker=dict(size=5, color=colors, opacity=0.8),
            text=[f"Type:{t}<br>Text:{d[:100]}..." for t, d in zip(doc_types, chroma_documents)],
            hoverinfo="text",
        )
    ]
)
fig.update_layout(
    title="2D Chroma Vector Store Visualization (Local Embedding)",
    scene=dict(xaxis_title="x", yaxis_title="y"),
    width=800,
    height=600,
    margin=dict(l=10, r=20, b=10, t=40),
)
fig.show()

In [62]:
folders = glob.glob("knowledge-base/*")
print(folders)

['knowledge-base/products', 'knowledge-base/contracts', 'knowledge-base/company', 'knowledge-base/employees']


In [68]:
tsne3 = TSNE(
    n_components=3,random_state=42
)
reduce_vectors_3d = tsne3.fit_transform(vectors)
fig3d = go.Figure(
    data = [
        go.Scatter3d(
            x=reduce_vectors_3d[:, 0],
            y=reduce_vectors_3d[:, 1],
            z=reduce_vectors_3d[:, 2],
            mode="markers",
            marker=dict(size=5,color=colors,opacity=0.8),
            text=[f"Type:{t}<br>Text:{d[:100]}..." for t, d in zip(doc_types, chroma_documents)],
            hoverinfo="text"
        )
    ]
)
fig3d.update_layout(
    title="3D Chroma Vector Store Visualization(BY LOcal Embedding)",
    scene= dict(xaxis_title="x",yaxis_title="y",zaxis_title="z"),
    width=900,
    height=700,
    margin=dict(r=10,b=10,l=10,t=40)
)
fig3d.show()